## SEC 13F Holdings as Latent Factor User–Item Interaction Matrix

### Entities

| Recommender System Term | Financial Interpretation |
|---|---|
| **User** | Institutional investor / fund manager (13F filer) |
| **Item** | Stock / security (CUSIP or ticker) |
| **Interaction** | Portfolio allocation of the institution to the security |

---

## Raw 13F Data Structure

| manager_id | cusip | ticker | shares | market_value | filing_date |
|---|---|---|---|---|---|

Each row indicates that manager \(i\) holds asset \(j\).

---

## Interaction Matrix

Construct the interaction matrix:

$$
R_{ij}
$$

Where

- $$(i) = institutional  manager$$
   
- $$(j) = asset (stock)$$

This matrix represents **institution–asset exposure**.

---

## Interaction Definition

### Portfolio Weight (commonly used)

$$
R_{ij} =
\frac{\text{market value}_{ij}}
{\sum_j \text{market value}_{ij}}
$$

Where

- \(R_{ij}\) represents the **portfolio allocation** of manager \(i\) to asset \(j\)

This produces a **normalized exposure matrix**.

---

## Resulting Matrix Example

| Manager / Stock | AAPL | MSFT | NVDA | JPM | XOM |
|---|---|---|---|---|---|
| Vanguard | 0.07 | 0.06 | 0.03 | 0.02 | 0.01 |
| BlackRock | 0.08 | 0.05 | 0.02 | 0.03 | 0.01 |
| Renaissance | 0.01 | 0.02 | 0.07 | 0.00 | 0.02 |

This forms the **user–item interaction matrix**.

---

## Latent Factor Model

Instead of computing explicit similarities, latent factor models approximate the interaction matrix as:

$$
R_{ij} \approx U_i^\top V_j
$$

Where

- \(U_i \in \mathbb{R}^k\) = latent representation of institution \(i\)  
- \(V_j \in \mathbb{R}^k\) = latent representation of asset \(j\)  
- \(k\) = number of latent factors

Matrix form:

$$
R \approx U V^\top
$$

Where

- \(U\) = institutional latent factor matrix  
- \(V\) = asset latent factor matrix  

---

## Interpretation of Latent Factors

The latent factors capture **hidden portfolio structure**, such as:

- sector preferences  
- factor exposures  
- growth vs value tilt  
- institutional strategy similarities

These factors are **learned from co-holding patterns** rather than predefined financial attributes.

---

## Predicted Interaction

The predicted allocation or affinity between institution \(i\) and asset \(j\) is:

$$
\hat{R}_{ij} = U_i^\top V_j
$$

Higher values indicate a **stronger latent preference**.

---

## Asset Similarity in Latent Space

Assets can be compared using their latent embeddings:

$$
sim(j,l) =
\frac{V_j \cdot V_l}
{\|V_j\| \|V_l\|}
$$

This captures **similar institutional ownership patterns**.

---

## Institutional Similarity in Latent Space

Similarly, institutions can be compared via:

$$
sim(i,k) =
\frac{U_i \cdot U_k}
{\|U_i\| \|U_k\|}
$$

This measures similarity in **portfolio construction behavior**.

# Black–Litterman Model

## 1. Market-Implied Equilibrium Returns

$$
\Pi = \delta \Sigma w_{mkt}
$$

**Where**

- $\Pi$ — implied equilibrium excess returns (prior expected returns)  
- $\delta$ — risk-aversion coefficient  
- $\Sigma$ — covariance matrix of asset returns  
- $w_{mkt}$ — market capitalization weights  

---

## 2. Investor Views Representation

$$
P\mu = Q
$$

**Where**

- $P$ — pick matrix specifying which assets participate in each view  
- $\mu$ — unknown true expected returns  
- $Q$ — vector of expected returns from investor views  

---

## 3. View Uncertainty

$$
\Omega
$$

**Where**

- $\Omega$ — covariance matrix representing uncertainty in investor views  

---

## 4. Prior Uncertainty Scaling

$$
\tau \Sigma
$$

**Where**

- $\tau$ — scalar representing uncertainty in the prior equilibrium returns  

---

## 5. Posterior Expected Returns (Black–Litterman Update)

$$
\mu_{BL} =
\left[
(\tau \Sigma)^{-1} + P^\top \Omega^{-1} P
\right]^{-1}
\left[
(\tau \Sigma)^{-1}\Pi + P^\top \Omega^{-1}Q
\right]
$$

**Where**

- $\mu_{BL}$ — posterior expected returns incorporating investor views  

---

## 6. Portfolio Weights

$$
w_{BL} = \frac{1}{\delta}\Sigma^{-1}\mu_{BL}
$$

**Where**

- $w_{BL}$ — optimal portfolio weights  
- $\delta$ — risk-aversion coefficient  

---

# Step-by-Step Procedure

### Step 1 — Estimate Covariance Matrix

$$
\Sigma
$$

from historical asset returns or a shrinkage estimator.

---

### Step 2 — Determine Market Portfolio Weights

$$
w_{mkt}
$$

---

### Step 3 — Compute Equilibrium Expected Returns

$$
\Pi = \delta \Sigma w_{mkt}
$$

---

### Step 4 — Specify Investor Views

Define:

- $P$ — pick matrix  
- $Q$ — expected return of each view  

---

### Step 5 — Specify View Confidence

$$
\Omega
$$

---

### Step 6 — Choose Prior Uncertainty Parameter

$$
\tau
$$

---

### Step 7 — Compute Posterior Expected Returns

$$
\mu_{BL}
$$

using the Black–Litterman update equation.

---

### Step 8 — Compute Optimal Portfolio Weights

$$
w_{BL} = \frac{1}{\delta}\Sigma^{-1}\mu_{BL}
$$

---

# Outputs

- Posterior expected returns: $ \mu_{BL} $  
- Optimal portfolio weights: $ w_{BL} $

In [1]:
import pandas as pd
import ast

In [21]:
files = [
    "stocktwits_0.csv",
    "stocktwits_1.csv"
    ]

In [22]:
import pandas as pd

df = pd.concat(
    [
        pd.read_csv(
            f,
            usecols=['user_id', 'created_at', 'sentiment', 'symbol_list']
        )
        for f in files
    ],
    ignore_index=True,
)

In [23]:
df = df[df['sentiment'].notnull()]
df = df[df["symbol_list"].notnull()]

In [24]:
df["symbol_list"] = (
    df["symbol_list"]
    .astype(str)
    .str.replace("[\\[\\]']", "", regex=True)
)

In [25]:
df = df[df["symbol_list"].notnull()]

In [26]:
df.loc[df["sentiment"] == "Bearish", "sentiment"] = -1
df.loc[df["sentiment"] == "Bullish", "sentiment"] = 1

In [27]:
df

,user_id,created_at,sentiment,symbol_list
0,6472,2012-10-15 18:57:06+00:00,-1,ZNGA
1,6472,2012-10-15 18:57:06+00:00,-1,META
2,148519,2012-10-15 18:57:38+00:00,1,FVI
3,75026,2012-10-15 18:57:39+00:00,1,GS
4,155028,2012-10-15 18:58:20+00:00,1,WYNN
...,...,...,...,...
13887279,256168,2014-02-09 02:19:32+00:00,1,AAPL
13887280,130129,2014-02-09 02:19:47+00:00,1,BB
13887281,304311,2014-02-09 02:21:52+00:00,1,BB
13887282,304311,2014-02-09 02:23:16+00:00,1,BB


In [28]:
df.user_id.nunique()

160455

In [29]:
df.symbol_list.nunique()

57682

In [ ]:
import requests

In [31]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://en.wikipedia.org/"
}

In [32]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
r = requests.get(url, headers=headers)
spx = pd.read_html(r.text)[0]

/tmp/ipykernel_186170/3083819388.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  spx = pd.read_html(r.text)[0]


In [33]:
spx_tickers = spx["Symbol"].tolist()

In [34]:
df = df[df["symbol_list"].isin(spx_tickers)]

In [36]:
df = (
    df.sort_values("created_at", ascending=False)
      .drop_duplicates(subset=["user_id", "symbol_list"], keep="first")
      .reset_index(drop=True)
)

In [40]:
df.to_csv("cleaned_ratings.csv")

In [2]:
import black_litterman_recommender as blr

In [3]:
df = pd.read_csv('cleaned_ratings.csv')

In [6]:
util = blr.stocktwits_to_sentiment_matrix(
    df,
    user_col = "user_id",
    asset_col = "symbol_list",
    sentiment_col = "sentiment",
    bullish_values = (1, True, "bullish", "Bullish", "BULLISH"),
    bearish_values = (-1, False, "bearish", "Bearish", "BEARISH"),
)


In [7]:
util

symbol_list,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,WST,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
user_id,,,,,,,,,,,,,,,,,,,,,
5,0,1,0,0,0,0,1,0,0,0,...,0,0,1,0,1,0,0,0,0,0
13,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
15,0,1,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
18,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
24,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2996987,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2997004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2997317,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
